# XGBoost hill-climb metalearner

Stack a stage-2 XGBoost metalearner on the accepted stage-1 hill-climbing ensemble models. Uses sampled Optuna search on stage-1 probability outputs as meta-features, full-fold cross-validation, and trains a final model on the full training set to produce the submission.

In [1]:
import os
import pickle
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

from helper_functions.data_preprocessing import encode_label
from hill_climbing_ensemble.ml_utils import XGBoostError, build_xgb_model, summarize_with_ci
from hill_climbing_ensemble.scoring import fit_model_from_spec

## 1. Run configuration

In [ ]:
# Inputs from upstream notebooks.
ENGINEERED_CV_FOLDS   = '../data/tmp/05-engineered-cv-folds.pkl'
ENGINEERED_TRAIN_DATA = '../data/tmp/05-engineered-train-data.csv'
ENGINEERED_TEST_DATA  = '../data/tmp/05-engineered-test-data.csv'
ENSEMBLE_PAYLOAD      = '../data/results/08-xgboost-ensemble.pkl'

# Outputs for this notebook.
OPTUNA_DB_FILE          = '../data/results/optuna-studies.db'
CROSS_VALIDATION_SCORES = '../data/results/09-xgboost-metalearner-scores.pkl'
FINAL_SUBMISSION_FILE   = '../data/submission.csv'

# Main run toggles.
RUN_OPTUNA_SEARCH         = True
RUN_FULL_CV_ESTIMATE      = False  # Set False for fast submission runs.
USE_BALANCED_CLASS_WEIGHT = True

# Number of stage-1 accepted models to use.
STAGE1_MODEL_COUNT = 11

# Optuna sampled search controls.
N_OPTUNA_TRIALS               = 60
SEARCH_FOLD_LIMIT             = 4
SEARCH_TRAIN_SAMPLE_FRAC      = 0.35
SEARCH_VALIDATION_SAMPLE_FRAC = 0.35
SEARCH_USE_SAMPLING           = True

# Full CV estimate controls.
ESTIMATE_FOLD_LIMIT             = None
ESTIMATE_USE_SAMPLING           = False
ESTIMATE_TRAIN_SAMPLE_FRAC      = 1.0
ESTIMATE_VALIDATION_SAMPLE_FRAC = 1.0

# Compute controls.
PARALLEL_GPU_IDS = (0, 1)
RANDOM_SEED      = 315


## 2. Load artifacts

In [ ]:
with open(ENGINEERED_CV_FOLDS, 'rb') as handle:
    engineered_folds = pickle.load(handle)

with open(ENSEMBLE_PAYLOAD, 'rb') as handle:
    ensemble_payload = pickle.load(handle)

train_df = pd.read_csv(ENGINEERED_TRAIN_DATA)
test_df  = pd.read_csv(ENGINEERED_TEST_DATA)

raw_train = pd.read_csv(
    'https://media.githubusercontent.com/media/gperdrizet/fullstack-2605/'
    'refs/heads/main/data/student-health-risk-train.csv'
)
_, label_encoder = encode_label(raw_train['health_condition'])

stage1_specs = ensemble_payload.get('accepted_specs', [])[:STAGE1_MODEL_COUNT]
n_classes    = len(label_encoder.classes_)

if len(stage1_specs) < STAGE1_MODEL_COUNT:
    raise ValueError(f'Expected {STAGE1_MODEL_COUNT} stage-1 models, found {len(stage1_specs)}.')

print(f'Loaded {len(engineered_folds)} engineered folds')
print(f'Stage-1 models loaded:  {len(stage1_specs)}')
print(f'Engineered train shape: {train_df.shape}')
print(f'Engineered test shape:  {test_df.shape}')
print(f'Classes: {list(label_encoder.classes_)}')

## 3. Build stage-1 meta-features

In [ ]:
def _align_probabilities(probabilities, model_classes):
    """Map model output columns to global class-index order."""

    aligned = np.zeros((len(probabilities), n_classes), dtype=float)
    classes_array = np.asarray(model_classes)

    class_ids = (
        classes_array.astype(int)
        if np.issubdtype(classes_array.dtype, np.number)
        else label_encoder.transform(classes_array.astype(str))
    )

    for col_idx, class_id in enumerate(class_ids):
        aligned[:, int(class_id)] = probabilities[:, col_idx]

    return aligned


def _build_meta_features(specs, x_train, y_train, x_predict, fold_seed):
    """Return stacked stage-1 probability matrices for train split and prediction set."""

    train_blocks, predict_blocks = [], []

    for model_idx, spec in enumerate(specs, start=1):
        model = fit_model_from_spec(spec, x_train=x_train, y_train=y_train, fold_seed=fold_seed + model_idx)
        train_blocks.append(_align_probabilities(model.predict_proba(x_train[spec['feature_columns']]), model.classes_))
        predict_blocks.append(_align_probabilities(model.predict_proba(x_predict[spec['feature_columns']]), model.classes_))

    return np.hstack(train_blocks), np.hstack(predict_blocks)

In [ ]:
%%time

meta_folds = []

for fold_idx, fold in enumerate(engineered_folds, start=1):
    y_tr_arr = np.asarray(fold['y_train'])
    y_va_arr = np.asarray(fold['y_validation'])
    y_train_ids = y_tr_arr if np.issubdtype(y_tr_arr.dtype, np.number) else label_encoder.transform(y_tr_arr.astype(str))
    y_val_ids   = y_va_arr if np.issubdtype(y_va_arr.dtype, np.number) else label_encoder.transform(y_va_arr.astype(str))

    x_train_meta, x_val_meta = _build_meta_features(
        stage1_specs,
        x_train=fold['x_train'],
        y_train=fold['y_train'],
        x_predict=fold['x_validation'],
        fold_seed=RANDOM_SEED + 1000 * fold_idx,
    )

    meta_folds.append({
        'x_train':      x_train_meta,
        'y_train':      np.asarray(y_train_ids),
        'x_validation': x_val_meta,
        'y_validation': np.asarray(y_val_ids),
    })

x_train_full = train_df.drop('health_condition', axis=1)
y_train_full = train_df['health_condition']
x_test_full  = test_df.drop('id', axis=1)

x_train_meta_full, x_test_meta_full = _build_meta_features(
    stage1_specs,
    x_train=x_train_full,
    y_train=y_train_full,
    x_predict=x_test_full,
    fold_seed=RANDOM_SEED + 50000,
)

y_tr_full_arr = np.asarray(y_train_full)

y_train_ids_full = (
    y_tr_full_arr.astype(int)
    if np.issubdtype(y_tr_full_arr.dtype, np.number)
    else label_encoder.transform(y_tr_full_arr.astype(str))
)

print(f'Meta folds prepared:   {len(meta_folds)}')
print(f'Meta feature count:    {x_train_meta_full.shape[1]}')
print(f'Full train meta shape: {x_train_meta_full.shape}')
print(f'Full test meta shape:  {x_test_meta_full.shape}')

## 4. Optuna sampled search

In [6]:
%%time

search_folds = meta_folds[:SEARCH_FOLD_LIMIT] if SEARCH_FOLD_LIMIT else meta_folds

def _objective(trial):
    params = {
        'max_depth':        trial.suggest_int('max_depth', 2, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 1e-3, 2e-1, log=True),
        'n_estimators':     trial.suggest_int('n_estimators', 200, 2600, step=100),
        'min_child_weight': trial.suggest_float('min_child_weight', 1.0, 20.0),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma':            trial.suggest_float('gamma', 1e-8, 3.0, log=True),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-6, 50.0, log=True),
    }

    fold_scores = []

    for fold_idx, fold in enumerate(search_folds, start=1):
        x_tr, y_tr = fold['x_train'], fold['y_train']
        x_va, y_va = fold['x_validation'], fold['y_validation']

        if SEARCH_USE_SAMPLING:
            rng = np.random.default_rng(RANDOM_SEED + trial.number * 100 + fold_idx)
            tr_idx = rng.choice(len(y_tr), size=max(1, int(len(y_tr) * SEARCH_TRAIN_SAMPLE_FRAC)), replace=False)
            va_idx = rng.choice(len(y_va), size=max(1, int(len(y_va) * SEARCH_VALIDATION_SAMPLE_FRAC)), replace=False)
            x_tr, y_tr = x_tr[tr_idx], y_tr[tr_idx]
            x_va, y_va = x_va[va_idx], y_va[va_idx]

        sample_weight = compute_sample_weight(class_weight='balanced', y=y_tr) if USE_BALANCED_CLASS_WEIGHT else None
        model = build_xgb_model(params, seed=RANDOM_SEED + fold_idx, prefer_gpu=True)
        
        try:
            model.fit(x_tr, y_tr, sample_weight=sample_weight)

        except XGBoostError:
            model = build_xgb_model(params, seed=RANDOM_SEED + fold_idx, prefer_gpu=False)
            model.fit(x_tr, y_tr, sample_weight=sample_weight)

        fold_scores.append(float(balanced_accuracy_score(y_va, model.predict(x_va))))

    trial.set_user_attr('fold_scores', fold_scores)
    trial.set_user_attr('mean', float(np.mean(fold_scores)))

    return float(np.median(fold_scores))


if RUN_OPTUNA_SEARCH:
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study = optuna.create_study(
        study_name='metalearner_09',
        storage='sqlite:///' + os.path.abspath(OPTUNA_DB_FILE),
        direction='maximize',
        load_if_exists=True,
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    )

    study.optimize(_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

else:
    study = optuna.load_study(
        study_name='metalearner_09',
        storage='sqlite:///' + os.path.abspath(OPTUNA_DB_FILE),
    )

best_params = study.best_trial.params

print(f'Trials completed: {len(study.trials)}')
print(f'Best sampled-CV median: {study.best_value:.5f}')
print('Best params:')

for key, value in best_params.items():
    print(f'  {key}: {value}')

In [ ]:
# Plot sorted trial median scores to show Optuna search distribution.
trial_values = sorted(
    [t.value for t in study.trials if t.value is not None],
    reverse=True,
)

plt.title('Optuna search trial scores (sorted)')
plt.plot(range(len(trial_values)), trial_values, color='black')
plt.xlabel('Trial rank')
plt.ylabel('Sampled-CV median balanced accuracy')
plt.tight_layout()
plt.show()

## 5. Cross-validation performance estimate

In [ ]:
if RUN_FULL_CV_ESTIMATE:
    full_fold_scores = []

    for fold_idx, fold in enumerate(meta_folds, start=1):
        x_tr, y_tr = fold['x_train'], fold['y_train']
        x_va, y_va = fold['x_validation'], fold['y_validation']

        sample_weight = compute_sample_weight(class_weight='balanced', y=y_tr) if USE_BALANCED_CLASS_WEIGHT else None
        model = build_xgb_model(best_params, seed=RANDOM_SEED + fold_idx, prefer_gpu=True)

        try:
            model.fit(x_tr, y_tr, sample_weight=sample_weight)

        except XGBoostError:
            model = build_xgb_model(best_params, seed=RANDOM_SEED + fold_idx, prefer_gpu=False)
            model.fit(x_tr, y_tr, sample_weight=sample_weight)

        full_fold_scores.append(float(balanced_accuracy_score(y_va, model.predict(x_va))))

    cv_results = {
        'params':                   best_params,
        'fold_scores':              full_fold_scores,
        'mean':                     float(np.mean(full_fold_scores)),
        'median':                   float(np.median(full_fold_scores)),
        'std':                      float(np.std(full_fold_scores)),
        'fold_count_used':          len(meta_folds),
        'fold_count_available':     len(engineered_folds),
        'use_sampling':             ESTIMATE_USE_SAMPLING,
        'use_balanced_class_weight': USE_BALANCED_CLASS_WEIGHT,
    }

    with open(CROSS_VALIDATION_SCORES, 'wb') as handle:
        pickle.dump(cv_results, handle)

else:
    if os.path.exists(CROSS_VALIDATION_SCORES):
        with open(CROSS_VALIDATION_SCORES, 'rb') as handle:
            cv_results = pickle.load(handle)
    else:
        cv_results = {
            'fold_scores': [],
            'mean': np.nan,
            'median': np.nan,
            'std': np.nan,
            'fold_count_used': 0,
            'fold_count_available': len(engineered_folds),
            'use_sampling': ESTIMATE_USE_SAMPLING,
            'use_balanced_class_weight': USE_BALANCED_CLASS_WEIGHT,
        }

ci_summary = summarize_with_ci(cv_results['fold_scores']) if cv_results['fold_scores'] else None
cv_results['mean_ci_95']   = ci_summary['mean'] if ci_summary is not None else {'value': np.nan, 'ci_lower': np.nan, 'ci_upper': np.nan}
cv_results['median_ci_95'] = ci_summary['median'] if ci_summary is not None else {'value': np.nan, 'ci_lower': np.nan, 'ci_upper': np.nan}

print('Cross-validation summary')
print(
    f"Mean balanced accuracy:   {cv_results['mean_ci_95']['value']:.4f} "
    f"(95% CI: {cv_results['mean_ci_95']['ci_lower']:.4f}, {cv_results['mean_ci_95']['ci_upper']:.4f})"
)

print(
    f"Median balanced accuracy: {cv_results['median_ci_95']['value']:.4f} "
    f"(95% CI: {cv_results['median_ci_95']['ci_lower']:.4f}, {cv_results['median_ci_95']['ci_upper']:.4f})"
)

print(f"Std balanced accuracy:    {cv_results['std']:.4f}")
print(f"Folds used: {cv_results['fold_count_used']}/{cv_results['fold_count_available']}")
print(f"Used balanced class weight: {cv_results.get('use_balanced_class_weight', False)}")


In [ ]:
plot_fold_scores = cv_results['fold_scores']

plt.title('Cross-validation balanced accuracy distribution')
sns.boxplot(
    x=plot_fold_scores,
    color='lightgray',
    boxprops={'facecolor': 'lightgray', 'edgecolor': 'black'},
    medianprops={'color': 'black', 'linewidth': 1.5},
    whiskerprops={'color': 'black'},
    capprops={'color': 'black'},
)
plt.xlabel('Balanced accuracy')
plt.tight_layout()
plt.show()

## 6. Train final model and generate submission

In [ ]:
final_sample_weight = None

if USE_BALANCED_CLASS_WEIGHT:
    final_sample_weight = compute_sample_weight(class_weight='balanced', y=y_train_ids_full)

final_model = build_xgb_model(best_params, seed=RANDOM_SEED, prefer_gpu=True)

try:
    final_model.fit(x_train_meta_full, y_train_ids_full, sample_weight=final_sample_weight)

except XGBoostError:
    final_model = build_xgb_model(best_params, seed=RANDOM_SEED, prefer_gpu=False)
    final_model.fit(x_train_meta_full, y_train_ids_full, sample_weight=final_sample_weight)

test_predictions = final_model.predict(x_test_meta_full).astype(int)

submission_df = pd.DataFrame({
    'id':               test_df['id'],
    'health_condition': label_encoder.inverse_transform(test_predictions),
})

submission_df.to_csv(FINAL_SUBMISSION_FILE, index=False)

print(f'Saved submission to {FINAL_SUBMISSION_FILE}')
print(submission_df['health_condition'].value_counts())
submission_df.head()

In [ ]:
import subprocess
from pathlib import Path

repo_root = Path('..').resolve()
submission_relpath = 'data/submission.csv'
submission_path = repo_root / submission_relpath
tag_name = 'v0.5.0'

if not submission_path.exists():
    raise FileNotFoundError(f'Missing submission file: {submission_path}')

def run_cmd(command):
    print('>', ' '.join(command))
    result = subprocess.run(
        command,
        cwd=repo_root,
        capture_output=True,
        text=True,
        check=False,
    )
    if result.stdout.strip():
        print(result.stdout.strip())

    if result.returncode != 0:
        if result.stderr.strip():
            print(result.stderr.strip())

        raise RuntimeError(f'Command failed ({result.returncode}): {" ".join(command)}')

    return result


# Stages both submission and notebook file present on disk.
run_cmd(['git', 'add', submission_relpath])
status = run_cmd(['git', 'status', '--porcelain', '--', submission_relpath])

if status.stdout.strip():
    run_cmd(['git', 'commit', '-m', 'submission: final metalearner predictions'])
    run_cmd(['git', 'push', 'origin', 'main'])

else:
    print('No new changes in submission.')

run_cmd(['git', 'tag', '-a', tag_name, '-m', f'Release {tag_name}'])
run_cmd(['git', 'push', 'origin', tag_name])

print(f'Release flow complete for tag {tag_name}.')

> git add data/submission.csv
> git status --porcelain -- data/submission.csv
No new changes in submission.
> git tag -a v0.4.6 -m Release v0.4.6
> git push origin v0.4.6
Release flow complete for tag v0.4.6.
